# Manual email categorization

Create the golden dataset by reading each email and assigning its correct category yourself.
No LLM is used in this notebook.

Select the Python kernel from `eval-genius/.venv` in VS Code.
Run cells individually from top to bottom. Do not use **Run All**: labeling requires your input,
and the correction cell is optional.

## Category rules

- **complaint**: reports an unresolved problem or requests resolution.
- **feedback**: shares praise or a suggestion without requesting resolution of a current problem.
  Appreciation after an issue has been resolved belongs here.
- **general_query**: asks for information without reporting a problem.

For mixed emails, prioritize complaint, then feedback, then general_query.
Read both the subject and body. A question about a missing refund can still be a complaint.

The intended authoring mix is 20 feedback, 15 complaints, and 15 general queries.
Use meaning to label the emails; do not force the counts to match.

The source stays in `docs/dataset/emails.json`.
Progress goes into `docs/goldends/manual_labels.json`.
After reviewing all 50 labels, export `docs/goldends/goldens.json`.

## 1. Set the paths

The project path below works regardless of the notebook's working directory.
Change it if you move the project. This cell only defines paths.

In [1]:
import json
from pathlib import Path
from collections import Counter

project_root = Path(
    "C:/a/pproj/AI/zzTwo/Evals2/core/eval-genius/"
    "1_email_categorization_eval"
)

dataset_path = project_root / "docs" / "dataset" / "emails.json"
goldens_dir = project_root / "docs" / "goldends"
progress_path = goldens_dir / "manual_labels.json"
golden_path = goldens_dir / "goldens.json"

categories = ["feedback", "complaint", "general_query"]

## 2. Load the emails and any saved labels

The first run starts an empty list. Later runs reload your progress.
If you restart the kernel, run steps 1 and 2 again before continuing.

In [2]:
emails = json.loads(dataset_path.read_text(encoding="utf-8"))

goldens_dir.mkdir(parents=True, exist_ok=True)

if progress_path.exists():
    goldens = json.loads(progress_path.read_text(encoding="utf-8"))
else:
    goldens = []

print(f"Total emails: {len(emails)}")
print(f"Already labelled: {len(goldens)}")
print(f"Remaining: {len(emails) - len(goldens)}")

Total emails: 50
Already labelled: 0
Remaining: 50


## 3. Label five emails

Run this cell and type a category into each VS Code input box.
After each valid label, your progress is saved automatically.

Run this same cell again for the next five emails. Continue until none remain.
Previously labelled emails are skipped. You can change `batch_size` if needed.

If you mistype a label, the assertion stops the cell. Run it again to continue
from the first unlabelled email. Earlier saved labels are preserved.

In [ ]:
batch_size = 5
labelled_ids = {item["id"] for item in goldens}
remaining = [email for email in emails if email["id"] not in labelled_ids]

for email in remaining[:batch_size]:
    print(f"\n--- {email['id']} ---")
    print(f"Subject: {email['subject']}\n")
    print(email["body"])

    category = input(
        "\nLabel (feedback / complaint / general_query): "
    ).strip().lower()

    assert category in categories, "Enter feedback, complaint, or general_query."

    goldens.append({
        **email,
        "expected_output": category
    })

    progress_path.write_text(
        json.dumps(goldens, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print(f"Saved: {email['id']} -> {category}")

print(f"\nLabelled {len(goldens)} of {len(emails)} emails.")


--- email_001 ---
Subject: Please stop the calls after delivery

Dear team,

I've received three calls from the delivery partner after my order was completed. None were about the order, and I've asked him not to call again. Please investigate this and make sure my contact details are not used this way.

Thank you.
Saved: email_001 -> complaint

--- email_002 ---
Subject: The packaging held up well

Hi,

My soup and noodles came through a fairly bumpy ride without spilling. The paper bag was easy to recycle too. Whoever decided to keep the hot and cold items apart got that right.

Thanks,
Samar


## 4. Review your labels

Run this whenever you want to check progress. Read each email's full text again
if the subject alone is not enough. The counts are a useful check, not a rule
that overrides your judgment.

In [ ]:
for item in goldens:
    print(f"{item['id']} | {item['expected_output']} | {item['subject']}")

counts = Counter(item["expected_output"] for item in goldens)

print("\nCategory counts:")
for category in categories:
    print(f"{category}: {counts[category]}")

print(f"\nTotal labelled: {len(goldens)} / {len(emails)}")

## 5. Correct a label (optional)

Skip this cell if your labels are correct.
Enter an email ID to read its full text, then enter the replacement category.
The correction is saved to your progress file. Run the review cell again afterward.

In [ ]:
email_id = input("Email ID to review, for example email_001: ").strip()
matches = [item for item in goldens if item["id"] == email_id]

assert len(matches) == 1, "Enter the ID of an email you have already labelled."

item = matches[0]

print(f"\nSubject: {item['subject']}\n")
print(item["body"])
print(f"\nCurrent label: {item['expected_output']}")

new_category = input(
    "Correct label (feedback / complaint / general_query): "
).strip().lower()

assert new_category in categories, "Enter feedback, complaint, or general_query."

item["expected_output"] = new_category

progress_path.write_text(
    json.dumps(goldens, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"Updated {email_id} -> {new_category}")

## 6. Save the reviewed golden dataset

Run this only when you have reviewed all labels.
It checks that every source email appears exactly once, its text is unchanged,
and its label is valid. It then saves the final file in source-email order.

Each record contains `id`, `subject`, `body`, and `expected_output`.
This is your human-reviewed reference for evaluating the LLM.

Running this cell again replaces the previous export with your current labels.
After any later correction, rerun the review and export cells.

In [ ]:
source_ids = [email["id"] for email in emails]
golden_ids = [item["id"] for item in goldens]

assert len(goldens) == len(emails), "Label every email before exporting."
assert len(set(golden_ids)) == len(golden_ids), "Duplicate email IDs found."
assert set(golden_ids) == set(source_ids), "Golden IDs must match the source emails."
assert all(item["expected_output"] in categories for item in goldens)

labels_by_id = {item["id"]: item for item in goldens}

for email in emails:
    labelled = labels_by_id[email["id"]]
    assert labelled["subject"] == email["subject"], "An email subject was changed."
    assert labelled["body"] == email["body"], "An email body was changed."

golden_dataset = [labels_by_id[email["id"]] for email in emails]

golden_path.write_text(
    json.dumps(golden_dataset, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"Saved {len(golden_dataset)} reviewed emails to:")
print(golden_path)